# ⚡ 05. Global LightGBM Forecaster with Tweedie Loss
**Capstone Project DS-02: Demand Forecasting & Inventory Optimization**

Trains a global tree-based ensemble with Tweedie objective ($p=1.15$) directly handling the zero-inflated, compound Poisson distribution of retail sales.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(str(Path.cwd().parent))
from src.data.data_loader import M5DataLoader
from src.features.build_features import FeaturePipeline
from src.models.lightgbm_model import LightGBMForecaster
from src.models.metrics import evaluate_forecast

loader = M5DataLoader(data_dir='../data/sample')
df = loader.build_merged_dataset(max_days=150)
pipeline = FeaturePipeline(lags=[7, 14, 28], rolling_windows=[7, 14])
feat_df = pipeline.fit_transform(df)
train_df, val_df = loader.split_train_val(feat_df, horizon_days=28)

X_train, y_train = pipeline.get_feature_matrix(train_df)
X_val, y_val = pipeline.get_feature_matrix(val_df)


In [ ]:
# Fit LightGBM
lgb = LightGBMForecaster(n_estimators=100, learning_rate=0.08, tweedie_variance_power=1.15)
lgb.fit(X_train, y_train)
preds = lgb.predict(X_val)

metrics = evaluate_forecast(y_val.values, preds['point'], y_train.values, y_lower=preds['q_0.10'], y_upper=preds['q_0.90'])
print('LightGBM (Tweedie) Validation Metrics:', metrics)
